# Lab 02-03: Delta Lake Pipeline

**Module:** 02 -- Data Preparation (14% of exam)  
**UI Sections:** SQL Editor | SQL Warehouses | Queries | Query History | Catalog  
**Estimated Time:** 45--60 minutes  
**Difficulty:** Intermediate

---

## What and Why

You now have extracted text (Lab 02-01) and a chunking strategy (Lab 02-02). This lab puts it all
together: build a **complete pipeline** from raw text to a queryable **Delta table of chunks** in
Unity Catalog. This table becomes the foundation for Vector Search in later modules.

Without this step, your chunks live only in memory. Writing them to Delta Lake gives you:

- **ACID transactions** -- no partial writes, no corrupted tables
- **Time travel** -- roll back to any previous version of your chunk table
- **Schema enforcement** -- every row has the same shape
- **Change Data Feed (CDF)** -- Vector Search uses this to auto-sync new chunks

---

## Mental Model

> **The Delta Lake pipeline is like a food processing plant.**
>
> Raw ingredients (documents) come in, get washed (cleaned), chopped (chunked), and packaged
> (written to Delta). The final packaged product sits on the shelf (Unity Catalog) ready for
> consumers (Vector Search, SQL queries).
>
> The Change Data Feed is like a stock-tracking system on the shelf: every time a new package
> lands or an old one is removed, CDF records the change so downstream consumers know exactly
> what is new.

---

## Exam Alert

**Change Data Feed (CDF) must be enabled on your Delta table for Vector Search Delta Sync to work.**
If the exam asks "what table property is required before creating a Delta Sync index?", the answer
is `delta.enableChangeDataFeed = true`.

---

## Learning Objectives

By the end of this lab you will be able to:

1. Build the full ingestion pipeline: extract, clean, chunk
2. Write chunks to a Delta table with the correct schema and CDF enabled
3. Query chunks using `spark.sql` and `%sql` magic
4. Navigate the **SQL Editor**, **SQL Warehouses**, and **Query History** in the Databricks UI
5. Verify your table in the **Catalog** UI with `DESCRIBE EXTENDED`

---

## Exam Objectives Covered

| Objective | Tested Here |
|-----------|-------------|
| Write chunks to Delta Lake / Unity Catalog | Steps 1--2 |
| Enable Change Data Feed for Vector Search | Step 2 |
| Query tables via SQL Editor and SQL Warehouses | Steps 3--4 |
| Use Catalog UI to verify table properties | Step 5 |

---

## Step 1: Build the Extract, Clean, Chunk Pipeline

We bring together the techniques from the previous two labs into a single, end-to-end function.
Think of this as assembling the conveyor belt in our food processing plant: raw documents go in
one end, clean chunks come out the other.

The pipeline has three stages:

1. **Extract** -- pull raw text from source documents
2. **Clean** -- normalize whitespace, remove boilerplate, strip special characters
3. **Chunk** -- split text into overlapping, token-counted segments

In [ ]:
import re
import uuid
from typing import List, Dict

# ---------------------------------------------------------------------------
# Stage 1: Extract -- simulate loading documents (in production, read from
# cloud storage, PDFs, web scraping results, etc.)
# ---------------------------------------------------------------------------
raw_documents = {
    "databricks_overview.txt": (
        "Databricks is a unified analytics platform.  \n\n"
        "It combines data engineering, data science, and machine learning \n"
        "on a single collaborative platform built on Apache Spark.\n\n"
        "The Lakehouse architecture unifies the best of data warehouses \n"
        "and data lakes.  Delta Lake provides ACID transactions,  \n"
        "scalable metadata handling, and schema enforcement.\n\n"
        "Unity Catalog is the governance layer for all data assets, \n"
        "including tables, models, and functions.  It provides \n"
        "fine-grained access control and data lineage tracking."
    ),
    "vector_search_guide.txt": (
        "Vector Search in Databricks lets you create search indexes \n"
        "on Delta tables.  There are two index types:\n\n"
        "1. Delta Sync Index -- automatically syncs with the source \n"
        "   Delta table using Change Data Feed.  When rows are added, \n"
        "   updated, or deleted, the index updates automatically.\n\n"
        "2. Direct Vector Access Index -- you manage the vectors \n"
        "   yourself via API calls.  This gives more control but \n"
        "   requires manual synchronization.\n\n"
        "For most RAG use cases, Delta Sync Index is recommended \n"
        "because it is simpler to maintain and always up to date."
    ),
    "model_serving_basics.txt": (
        "Model Serving in Databricks allows you to deploy ML models \n"
        "as REST API endpoints.  Supported model types include:\n\n"
        "- Custom models registered in MLflow\n"
        "- Foundation models (e.g., DBRX, Llama, Mixtral)\n"
        "- External models (e.g., OpenAI, Anthropic via AI Gateway)\n\n"
        "Each serving endpoint has a configurable compute size and \n"
        "can auto-scale based on traffic.  You can also set up \n"
        "A/B testing with traffic splitting between model versions."
    ),
}

print(f"Loaded {len(raw_documents)} raw documents.")
for name, text in raw_documents.items():
    print(f"  {name}: {len(text)} chars")

In [ ]:
# ---------------------------------------------------------------------------
# Stage 2: Clean -- normalize whitespace, collapse blank lines, strip edges
# ---------------------------------------------------------------------------
def clean_text(text: str) -> str:
    """Wash the raw ingredients: normalize whitespace and remove noise."""
    text = text.replace("\r\n", "\n")          # normalize line endings
    text = re.sub(r"[ \t]+", " ", text)          # collapse horizontal whitespace
    text = re.sub(r"\n{3,}", "\n\n", text)       # max two consecutive newlines
    text = text.strip()
    return text


cleaned_documents = {name: clean_text(text) for name, text in raw_documents.items()}

print("Cleaned documents:")
for name, text in cleaned_documents.items():
    print(f"  {name}: {len(text)} chars (was {len(raw_documents[name])})")

In [ ]:
# ---------------------------------------------------------------------------
# Stage 3: Chunk -- split cleaned text into overlapping segments
# ---------------------------------------------------------------------------
def estimate_tokens(text: str) -> int:
    """Rough token estimate: ~4 characters per token for English text."""
    return max(1, len(text) // 4)


def chunk_text(
    text: str,
    source_doc: str,
    max_chars: int = 300,
    overlap_chars: int = 50,
) -> List[Dict]:
    """Chop the cleaned text into overlapping chunks with metadata."""
    chunks = []
    start = 0
    idx = 0
    while start < len(text):
        end = start + max_chars
        chunk_str = text[start:end]
        chunks.append({
            "chunk_id": str(uuid.uuid4()),
            "source_doc": source_doc,
            "chunk_text": chunk_str,
            "chunk_index": idx,
            "token_count": estimate_tokens(chunk_str),
        })
        start += max_chars - overlap_chars
        idx += 1
    return chunks


# Run the full pipeline: extract -> clean -> chunk
all_chunks = []
for doc_name, cleaned_text in cleaned_documents.items():
    doc_chunks = chunk_text(cleaned_text, source_doc=doc_name)
    all_chunks.extend(doc_chunks)

print(f"Pipeline produced {len(all_chunks)} chunks from {len(raw_documents)} documents.")
print()
# Show the first two chunks as a sanity check
for c in all_chunks[:2]:
    print(f"[{c['source_doc']}] chunk {c['chunk_index']} "
          f"({c['token_count']} tokens): {c['chunk_text'][:80]}...")

---

## Step 2: Write Chunks to a Delta Table with CDF Enabled

Now we package our chunks and put them on the shelf. In Databricks, "the shelf" is a
**Unity Catalog table** stored in **Delta Lake format**.

### Why Change Data Feed (CDF)?

Vector Search Delta Sync works by reading the *change feed* of your table. Every time you
INSERT, UPDATE, or DELETE rows, Delta Lake records those changes in a side-car log. Vector
Search reads that log to know exactly which embeddings to add, update, or remove from the
index.

Without CDF enabled, Delta Sync cannot track changes and will **refuse to create the index**.

### Schema

| Column | Type | Description |
|--------|------|-------------|
| `chunk_id` | STRING | Unique identifier for each chunk |
| `source_doc` | STRING | Name of the source document |
| `chunk_text` | STRING | The actual text content |
| `chunk_index` | INT | Position of the chunk within its source document |
| `token_count` | INT | Estimated token count for the chunk |

In [ ]:
# ---------------------------------------------------------------------------
# Convert chunks to a Spark DataFrame and write to Delta
# ---------------------------------------------------------------------------
# NOTE: On a Databricks cluster, `spark` is pre-configured. Locally, you
# would need to create a SparkSession yourself.

TABLE_NAME = "main.genai_labs.document_chunks"

# -- Databricks cluster code (uncomment when running on Databricks) ---------
# from pyspark.sql.types import StructType, StructField, StringType, IntegerType
#
# schema = StructType([
#     StructField("chunk_id",    StringType(), False),
#     StructField("source_doc",  StringType(), False),
#     StructField("chunk_text",  StringType(), False),
#     StructField("chunk_index", IntegerType(), False),
#     StructField("token_count", IntegerType(), False),
# ])
#
# df_chunks = spark.createDataFrame(all_chunks, schema=schema)
#
# # Write to Delta -- overwrite so the lab is idempotent
# df_chunks.write.format("delta").mode("overwrite").saveAsTable(TABLE_NAME)
#
# print(f"Wrote {df_chunks.count()} chunks to {TABLE_NAME}")
# --------------------------------------------------------------------------

# Local simulation using pandas (for study / review without a cluster)
import pandas as pd

df_chunks = pd.DataFrame(all_chunks)
print(f"DataFrame ready with {len(df_chunks)} rows.")
print(f"Schema: {list(df_chunks.columns)}")
print(f"\nTarget table: {TABLE_NAME}")
print()
df_chunks.head()

In [ ]:
# ---------------------------------------------------------------------------
# Enable Change Data Feed on the table
# ---------------------------------------------------------------------------
# IMPORTANT: This is the command the exam tests.
#
# On Databricks, run this after creating the table:
#
#   spark.sql("""
#       ALTER TABLE main.genai_labs.document_chunks
#       SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
#   """)
#
# You can also enable CDF at table creation time:
#
#   CREATE TABLE main.genai_labs.document_chunks (
#       chunk_id    STRING NOT NULL,
#       source_doc  STRING NOT NULL,
#       chunk_text  STRING NOT NULL,
#       chunk_index INT    NOT NULL,
#       token_count INT    NOT NULL
#   )
#   USING DELTA
#   TBLPROPERTIES (delta.enableChangeDataFeed = true);
#
# TIP: You can also enable CDF by default for ALL new tables in a schema:
#   ALTER SCHEMA main.genai_labs
#   SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

alter_cmd = f"ALTER TABLE {TABLE_NAME} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)"
print("Command to enable CDF:")
print(f"  {alter_cmd}")
print()
print("Why this matters:")
print("  - Vector Search Delta Sync reads the change feed to stay in sync.")
print("  - Without CDF, Delta Sync CANNOT track inserts, updates, or deletes.")
print("  - The exam WILL ask about this property.")

---

## Step 3: Query Chunks Using spark.sql and %sql Magic

Once your chunks are in a Delta table, anyone with access can query them -- data engineers,
data scientists, or downstream pipelines. In Databricks, you have two common ways to query:

1. **`spark.sql("...")`** in a Python cell -- great for programmatic access
2. **`%sql` magic** in a notebook cell -- great for quick exploration

### UI Navigation: SQL Editor and SQL Warehouses

In the Databricks workspace:

- **SQL Editor** (left sidebar, under "SQL"): A dedicated query editor for writing and
  running SQL statements. Think of it as a lightweight SQL IDE inside Databricks.
- **SQL Warehouses** (under "SQL" > "SQL Warehouses"): The serverless compute that
  executes your SQL queries. Like a vending machine -- you feed it a query, it returns
  results. You do not manage the cluster yourself.

The SQL Editor connects to a SQL Warehouse automatically. You select which warehouse to
use from a dropdown at the top of the editor.

In [ ]:
# ---------------------------------------------------------------------------
# Query using spark.sql (Python approach)
# ---------------------------------------------------------------------------
# On Databricks:
#
# result = spark.sql(f"SELECT * FROM {TABLE_NAME} LIMIT 10")
# result.show(truncate=50)
#
# Useful aggregate queries:
#
# spark.sql(f"""
#     SELECT source_doc,
#            COUNT(*)       AS num_chunks,
#            AVG(token_count) AS avg_tokens,
#            SUM(token_count) AS total_tokens
#     FROM {TABLE_NAME}
#     GROUP BY source_doc
#     ORDER BY num_chunks DESC
# """).show()

# Local simulation
print(f"Query: SELECT * FROM {TABLE_NAME} LIMIT 10")
print()
print(df_chunks.head(10).to_string(index=False))
print()

# Aggregate: chunks per document
print("\nChunks per document:")
summary = df_chunks.groupby("source_doc").agg(
    num_chunks=("chunk_id", "count"),
    avg_tokens=("token_count", "mean"),
    total_tokens=("token_count", "sum"),
)
print(summary.to_string())

In [ ]:
# ---------------------------------------------------------------------------
# Query using %sql magic (SQL approach)
# ---------------------------------------------------------------------------
# In a Databricks notebook, you would use a SQL cell:
#
# %sql
# SELECT source_doc,
#        COUNT(*)         AS num_chunks,
#        ROUND(AVG(token_count), 1) AS avg_tokens
# FROM main.genai_labs.document_chunks
# GROUP BY source_doc
# ORDER BY num_chunks DESC;
#
# The %sql magic sends the query to the attached SQL Warehouse.
# Results appear as an interactive table with sorting and filtering.

print("In a Databricks notebook, create a SQL cell with:")
print()
print("  %sql")
print("  SELECT source_doc,")
print("         COUNT(*)         AS num_chunks,")
print("         ROUND(AVG(token_count), 1) AS avg_tokens")
print(f"  FROM {TABLE_NAME}")
print("  GROUP BY source_doc")
print("  ORDER BY num_chunks DESC;")

---

## Step 4: Save Queries and Check Query History

### Saving Queries

In the **SQL Editor**, after you write and run a query:

1. Click the **Save** button (top-right of the editor)
2. Give it a descriptive name, e.g., `Chunk Summary by Document`
3. The saved query appears under **Queries** in the left sidebar

Saved queries are like bookmarks for SQL -- you can share them with teammates, schedule
them to run on a cadence, or attach them to dashboards.

### Query History

Navigate to **SQL** > **Query History** in the left sidebar. Here you can see:

- Every query you (or your warehouse) have executed
- Execution time, status, rows returned
- The warehouse that ran each query

This is useful for debugging slow queries and auditing who queried what.

In [ ]:
# ---------------------------------------------------------------------------
# Walkthrough: save a query and check history
# ---------------------------------------------------------------------------
# These are UI-based steps. Here is the exact path:
#
# 1. Open the Databricks workspace.
# 2. Left sidebar -> SQL -> SQL Editor.
# 3. Select your SQL Warehouse from the dropdown (top of editor).
# 4. Paste this query:
#
#    SELECT source_doc,
#           COUNT(*) AS num_chunks,
#           SUM(token_count) AS total_tokens
#    FROM main.genai_labs.document_chunks
#    GROUP BY source_doc;
#
# 5. Click "Run" (Ctrl+Enter / Cmd+Enter).
# 6. Click "Save" -> name it "Chunk Summary by Document".
# 7. Left sidebar -> SQL -> Query History -> find your execution.

print("UI Navigation:")
print("  1. Left sidebar -> SQL -> SQL Editor")
print("  2. Select SQL Warehouse from dropdown")
print("  3. Write and run your query")
print("  4. Save button -> name the query")
print("  5. Left sidebar -> SQL -> Queries (to find saved queries)")
print("  6. Left sidebar -> SQL -> Query History (to see all executions)")

---

## Step 5: Verify the Table in Catalog UI

The **Catalog** UI is your single pane of glass for all data assets in Databricks. Think
of it like a library card catalog: you can browse catalogs, schemas, and tables, inspect
columns, view sample data, and check table properties.

### Navigation Path

Left sidebar > **Catalog** > expand `main` > expand `genai_labs` > click `document_chunks`

You will see:
- **Schema** tab: column names and types
- **Sample Data** tab: first few rows
- **Details** tab: table format (Delta), location, properties

### Programmatic Verification with DESCRIBE EXTENDED

The `DESCRIBE EXTENDED` command shows all metadata including Delta properties.

In [ ]:
# ---------------------------------------------------------------------------
# Verify the table with DESCRIBE EXTENDED
# ---------------------------------------------------------------------------
# On Databricks:
#
# spark.sql(f"DESCRIBE EXTENDED {TABLE_NAME}").show(100, truncate=False)
#
# Look for these lines in the output:
#   - Provider: delta
#   - Table Properties: [delta.enableChangeDataFeed=true, ...]
#
# You can also check just the properties:
#
# spark.sql(f"SHOW TBLPROPERTIES {TABLE_NAME}").show(truncate=False)

print(f"DESCRIBE EXTENDED {TABLE_NAME}")
print()
print("Expected output (key rows):")
print("-" * 65)
print(f"{'col_name':<30} {'data_type':<20} {'comment':<15}")
print("-" * 65)
print(f"{'chunk_id':<30} {'string':<20} {''}")
print(f"{'source_doc':<30} {'string':<20} {''}")
print(f"{'chunk_text':<30} {'string':<20} {''}")
print(f"{'chunk_index':<30} {'int':<20} {''}")
print(f"{'token_count':<30} {'int':<20} {''}")
print(f"{'# Detailed Table Information':<30} {'---':<20} {''}")
print(f"{'Provider':<30} {'delta':<20} {''}")
print(f"{'Table Properties':<30} {'[delta.enableChangeDataFeed=true]':<20}")
print("-" * 65)
print()
print("Key things to verify:")
print("  1. Provider is 'delta' (not parquet, not csv)")
print("  2. Table Properties includes delta.enableChangeDataFeed=true")
print("  3. All five columns are present with correct types")

In [ ]:
# ---------------------------------------------------------------------------
# Final verification: row count and sample
# ---------------------------------------------------------------------------
# On Databricks:
#
# count = spark.sql(f"SELECT COUNT(*) AS total FROM {TABLE_NAME}").first()["total"]
# print(f"Total chunks in table: {count}")
#
# spark.sql(f"""
#     SELECT chunk_id, source_doc, chunk_index, token_count,
#            LEFT(chunk_text, 80) AS chunk_preview
#     FROM {TABLE_NAME}
#     ORDER BY source_doc, chunk_index
#     LIMIT 5
# """).show(truncate=False)

# Local simulation
print(f"Total chunks in table: {len(df_chunks)}")
print()
preview = df_chunks.copy()
preview["chunk_preview"] = preview["chunk_text"].str[:80] + "..."
print(preview[["chunk_id", "source_doc", "chunk_index", "token_count", "chunk_preview"]]
      .sort_values(["source_doc", "chunk_index"])
      .head(5)
      .to_string(index=False))

---

## Exam Tips

| # | Tip | Why It Matters |
|---|-----|----------------|
| 1 | CDF is enabled via `ALTER TABLE ... SET TBLPROPERTIES (delta.enableChangeDataFeed = true)` | Required before creating a Delta Sync Vector Search index |
| 2 | SQL Warehouses are the compute for SQL Editor queries -- separate from all-purpose clusters | The exam distinguishes between cluster types |
| 3 | `DESCRIBE EXTENDED` shows table format and properties | Used to verify Delta format and CDF status |
| 4 | Saved queries live under the **Queries** section of the SQL sidebar | Know the UI layout for exam questions about the workspace |
| 5 | Unity Catalog uses a three-level namespace: `catalog.schema.table` | The exam may ask about namespace structure (e.g., `main.genai_labs.document_chunks`) |

---

## Key Takeaways

### Pipeline Architecture
- A Delta Lake pipeline for RAG follows three stages: **extract, clean, chunk**
- The output is a Delta table in Unity Catalog with a well-defined schema
- This table is the **single source of truth** for all downstream consumers

### Change Data Feed (CDF)
- **CDF must be enabled** before creating a Vector Search Delta Sync index
- Enable it with `ALTER TABLE ... SET TBLPROPERTIES (delta.enableChangeDataFeed = true)`
- CDF tracks row-level changes (inserts, updates, deletes) so downstream indexes stay in sync

### SQL Workflow in Databricks
- **SQL Editor**: write and run SQL queries
- **SQL Warehouses**: serverless compute that executes SQL (separate from clusters)
- **Queries**: saved SQL statements you can share and schedule
- **Query History**: audit log of all executed queries
- **Catalog**: browse and verify tables, columns, and properties

---

## Next Lab

**Lab 02-04: Retrieval Evaluation** -- Now that your chunks are in a Delta table, how do
you know if your retriever is finding the right ones? Next, we measure retrieval quality
with Precision@K, Recall@K, MRR, and nDCG.